In [ ]:
import requests
import pandas as pd

def get_acumatica_table(url: str, username: str, password: str, params: dict = {"$format": "json"}) -> pd.DataFrame:
    """returns data table from odata url in acumatica"""

    resp = requests.get(url, auth=(username, password), params=params)
    resp.raise_for_status()

    data = resp.json().get("value", [])
    return pd.DataFrame(data)


URL = r"***"
USERNAME = "***"
PASSWORD = "***"
params = {"$select": "CustomerID,AccountGroup,CustomerName,PostalCode,State", "$format":"json"}

acu_cust_raw = get_acumatica_table(url=URL, username=USERNAME, password=PASSWORD, params=params)
acu_cust_raw


In [ ]:
import pandas as pd

pos_path = r"***"
sheet_name = "RawData"
pos_cust_raw = pd.read_excel(io=pos_path, sheet_name=sheet_name)
cols = ["SoldToName", "BillToCustomerZip", "BillToCustomerState"]
pos_cust = pos_cust_raw[cols]


In [ ]:


acu_out = (
    acu_cust_raw
    .assign(**{c: lambda d, c=c: d[c].astype("string").str.strip() for c in acu_cust_raw.columns})
    .pipe(add_postal, src_col="PostalCode", primary_name="ZipPrimary", drop_original=True)
    .rename(columns={"CustomerName": "Name", "State": "BillToState", "ZipPrimary":"BillToZip"})
)

pos_out = (
    pos_cust
    .assign(**{c: lambda d, c=c: d[c].astype("string").str.strip() for c in pos_cust.columns})
    .pipe(add_postal, src_col="BillToCustomerZip", primary_name="ZipPrimary", drop_original=True)
    .rename(columns={"SoldToName": "Name", "BillToCustomerState": "BillToState", "ZipPrimary": "BillToZip"})
)

# need to also drop duplicates in customer names. figure out how direct integrates with this process

In [ ]:
yaml_cfg = {
    'keep_cols': ['abc', 'def', 'xyz'],
    'numeric_cols': ['abc'],
    'str_cols': ['def'],
    'float_cols': ['xyz'],
    'date_cols': [], 
    'rename_map': {'ABC': 'abc', 'DEF': 'def', 'XYZ': 'xyz'},
}

[]